In [1]:
import numpy as np
import matplotlib.pyplot as plt
plt.ion()
from numba import cuda
import time

In [2]:
#_G = 6.6743/(10**11)
_G = 10

In [3]:
dt = 0.001
n = 1000

In [4]:
mass =  ((np.random.random_sample(n))*200+1).astype(np.float64)
coor =  ((np.random.random_sample((n,3))-0.5)*1000).astype(np.float64)
dcoor = ((np.random.random_sample((n,3))-0.5)*1000).astype(np.float64)

In [21]:
@cuda.jit    
def gpu(mass ,coor ,dcoor ,n ,coor_output ,dcoor_output ,debug):
	global _G,dt
    
	tid = cuda.grid(1)
	move_treads_ter = int(n*(n-1)/2)
	
	if tid>=move_treads_ter :	
		i = tid - move_treads_ter
		for x in range (3):   
			#coor_output[i,x] = coor[i,x] + dcoor[i,x]*dt
			cuda.atomic.add(coor ,(i,x) ,float(dcoor[i,x]*dt))
		return
    
	itt=n-1
	while tid>=itt :
		tid-=itt
		itt-=1
	i = int(itt	)
	j = int(tid)
    
	
    
	r2=0
	for x in range (3):
		tmp = (coor[j,x]-coor[i,x])
		r2 += tmp*tmp
        
	tmp = _G*mass[i]*mass[j]*(r2**-1.5)
    
	if(tmp>1000000):
		tmp=1000000
    
	for x in range (3):   
		fdt = tmp*(coor[j,x]-coor[i,x])*dt
		if i==1 :
			debug[j,x] = tmp
		#dcoor_output[j,x] = dcoor[j,x] - f*dt*mass(j)
		cuda.atomic.add(dcoor ,(j,x) , -fdt/mass[j] )
		#dcoor_output[i,x] = dcoor[i,x] + f*dt
		cuda.atomic.add(dcoor ,(i,x) , +fdt/mass[i] )

In [22]:
mass_gpu = cuda.to_device(mass)
coor_gpu = cuda.to_device(coor)
coor_gpu_output = cuda.device_array_like(coor_gpu)
dcoor_gpu = cuda.to_device(dcoor)
dcoor_gpu_output = cuda.device_array_like(dcoor_gpu)

debug= np.zeros((n,3))

In [23]:
coor_gpu

In [24]:
tpb = 100
for i in range(1000):
        
        start = time.time()
        gpu[int(n*(n+1)/2/tpb),tpb](mass_gpu,coor_gpu,dcoor_gpu,n,coor_gpu_output,dcoor_gpu_output,debug)
        end = time.time()
        print(end - start)
	
        cuda.synchronize()
        #coor_gpu = coor_gpu_output
        #dcoor_gpu = dcoor_gpu_output 
        coor = coor_gpu.copy_to_host()
        dcoor =   dcoor_gpu.copy_to_host()

0.4788322448730469
0.006018638610839844
0.006018638610839844
0.006018400192260742
0.006018638610839844
0.005984306335449219
0.0060193538665771484
0.006018877029418945
0.0069806575775146484
0.005984306335449219
0.00598454475402832
0.00694727897644043
0.006018400192260742
0.005986213684082031
0.005984067916870117
0.006017923355102539
0.006018877029418945
0.005983114242553711
0.005982398986816406
0.006018400192260742
0.005984783172607422
0.006019115447998047
0.006020069122314453
0.005983591079711914
0.005982875823974609
0.006947040557861328
0.0060176849365234375
0.006020069122314453
0.005982637405395508
0.005982398986816406
0.006018638610839844
0.0050220489501953125
0.006017923355102539
0.005961418151855469
0.006006717681884766
0.005984306335449219
0.005023479461669922
0.004986763000488281
0.006021261215209961
0.005982398986816406
0.005983829498291016
0.005020856857299805
0.005023479461669922
0.0049855709075927734
0.005021095275878906
0.006021022796630859
0.005983591079711914
0.0059852600

0.005986452102661133
0.004984140396118164
0.005983829498291016
0.00602269172668457
0.004967927932739258
0.005021095275878906
0.00502324104309082
0.005011796951293945
0.005984067916870117
0.004986763000488281
0.004973411560058594
0.005950450897216797
0.005984067916870117
0.005986213684082031
0.004984617233276367
0.005026578903198242
0.004986763000488281
0.005021095275878906
0.0049839019775390625
0.0049474239349365234
0.0050220489501953125
0.0050241947174072266
0.0059833526611328125
0.0050258636474609375
0.0049860477447509766
0.0059528350830078125
0.005983591079711914
0.004984140396118164
0.004987001419067383
0.005013942718505859
0.004988193511962891
0.005026102066040039
0.00594782829284668
0.006017923355102539
0.005985260009765625
0.005015373229980469
0.005020856857299805
0.005021572113037109
0.00601649284362793
0.005984067916870117
0.005950450897216797
0.005984783172607422
0.0050106048583984375
0.005957841873168945
0.006021261215209961
0.0059816837310791016
0.0059506893157958984
0.0050

0.0050122737884521484
0.004985809326171875
0.005024909973144531
0.006020545959472656
0.004985332489013672
0.004988193511962891
0.005011320114135742
0.005983591079711914
0.005959272384643555
0.004987239837646484
0.005010366439819336
0.004987239837646484
0.004986763000488281
0.004985809326171875
0.0059833526611328125
0.005013227462768555
0.005014181137084961
0.004988908767700195
0.0050203800201416016
0.005019426345825195
0.005012989044189453
0.004984855651855469
0.005013704299926758
0.0050144195556640625
0.005011558532714844
0.004987478256225586
0.0049855709075927734
0.0050201416015625
0.005017280578613281
0.005011320114135742
0.0050127506256103516
0.005984067916870117
0.005983591079711914
0.0059850215911865234
0.005024433135986328
0.004961729049682617
0.005984067916870117
0.005983829498291016
0.006011247634887695
0.005956888198852539
0.005986213684082031
0.005980253219604492
0.0059850215911865234
0.005984783172607422
0.005012035369873047
0.005013227462768555
0.005010843276977539
0.00498

KeyboardInterrupt: 